# BNPL Governance Workshop - Part 3: Tableflow

This notebook picks up where `kredivo-02.ipynb` left off. Part 2 produced several derived topics
with Flink SQL -- `orders_deduped_<you>`, `orders_high_value_<you>`, and `orders_enriched_<you>`
(orders joined with product name/category/price). Those topics are only reachable with a Kafka
client. Here we use **Tableflow** to turn `orders_enriched_<you>` into an **Apache Iceberg table**
that any Iceberg-aware engine -- DuckDB, Snowflake, Spark, Trino, Athena -- can query directly,
with zero Kafka client involved.

**What Tableflow actually does:** it takes the Avro records already flowing through a topic
(materialized from Confluent's Kora storage layer), converts them to Parquet files in object
storage, and continuously maintains Iceberg (or Delta Lake) metadata on top of those files. The
table it produces is a first-class Iceberg table -- readable by anything that speaks the Iceberg
REST Catalog spec, not a Confluent-proprietary export.

In this notebook we will:

1. Enable Tableflow on `orders_enriched_<you>`, using Confluent-managed storage (no S3 bucket of
   your own to configure).
2. Poll until the table has materialized and inspect its Iceberg snapshot metadata.
3. Query it **directly from this notebook**, from Python, via Confluent's built-in Iceberg REST
   Catalog -- first with `pyiceberg`, then with `duckdb` (the same two lines a data analyst on a
   laptop with no Confluent CLI would run).
4. Show (reference-only, since we don't have live Snowflake/Athena/Spark accounts in this
   workshop) the equivalent connection snippets for **Snowflake**, **Amazon Athena / Trino**, and
   **Apache Spark**.
5. Clean up -- optionally tearing down not just Tableflow, but everything the whole
   workshop created across Parts 1-3 (all topics, schema subjects, and Flink statements).

**Requirements, in addition to the `.env` values `kredivo-01.ipynb`/`kredivo-02.ipynb` already
use:**

- **Confluent CLI installed and logged in** (`confluent login`) for the *enable / status / disable*
  steps (`confluent tableflow topic ...`). These are control-plane operations the CLI already knows
  how to do correctly; if the CLI isn't available or isn't logged in, those cells print the exact
  command they would have run and explain what to expect, instead of failing.
- **A Tableflow-scoped API key/secret**, set as `TABLEFLOW_API_KEY` / `TABLEFLOW_API_SECRET` in
  `.env`, for the *query* steps (pyiceberg / DuckDB). This is a **different key from the Kafka,
  Schema Registry, and Flink keys** already in `.env` -- create one via Confluent Cloud Console ->
  your environment -> **Tableflow** tab -> **API Keys** -> **+ Add API key** (or
  `confluent api-key create --resource <environment-id>`). Without it, the query cells print the
  Iceberg REST Catalog endpoint and the SQL/Python you'd run once you have a key, instead of
  failing.

### Install dependencies

`pyiceberg` and `duckdb` are the two engines we use to query the Iceberg table directly from this
notebook; everything else is shared with Part 1/2.

**Example:** after this runs, `from pyiceberg.catalog.rest import RestCatalog` and `import duckdb`
succeed in the next cells.

In [ ]:
RUN_IN_NOTEBOOK = True  # Set to False if you'd rather copy PIP_CMD and run it yourself in a terminal (with .venv activated)

PIP_CMD = 'pip install -q "pyiceberg[pyarrow]" duckdb confluent-kafka python-dotenv pandas requests'

if RUN_IN_NOTEBOOK:
    !{PIP_CMD}
else:
    print(f"Skipping -- run this yourself in a terminal (with .venv activated):\n\n  $ {PIP_CMD}")

### Load credentials and rebuild this participant's topic names

Same derivation as `kredivo-02.ipynb`: `PARTICIPANT` namespaces every resource so this notebook
doesn't collide with other attendees. `TOPIC_ENRICHED` must already exist and have data in it --
run `kredivo-02.ipynb` (through Step 6) first if you haven't.

We also read the four Cloud-topology values already in `.env` from Part 2 (`ORG_ID`, `ENV_ID`,
`CLOUD_REGION`, `CLOUD_PROVIDER`) to construct the Iceberg REST Catalog endpoint -- Tableflow's
catalog URL is fully determined by those four values, no extra lookup needed.

In [ ]:
import os
import json as pyjson
import time
import shutil
import subprocess
import requests
import pandas as pd
from dotenv import load_dotenv

OK, FAIL, WARN, INFO = "✅", "❌", "⚠️", "ℹ️"

load_dotenv(".env")

FLINK_POOL = os.getenv("FLINK_COMPUTE_POOL_ID", "")
CLOUD_REGION = os.getenv("CLOUD_REGION", "").lower()
CLOUD_PROVIDER = os.getenv("CLOUD_PROVIDER", "").lower()
ORG_ID = os.getenv("ORG_ID", "")
ENV_ID = os.getenv("ENV_ID", "")
CLUSTER_ID = os.getenv("CONFLUENT_CLUSTER_ID", "")

TABLEFLOW_API_KEY = os.getenv("TABLEFLOW_API_KEY", "")
TABLEFLOW_API_SECRET = os.getenv("TABLEFLOW_API_SECRET", "")
TABLEFLOW_QUERY_AVAILABLE = all([CLOUD_REGION, CLOUD_PROVIDER, ORG_ID, ENV_ID, TABLEFLOW_API_KEY, TABLEFLOW_API_SECRET])

PARTICIPANT = os.getenv("PARTICIPANT_ID") or os.getenv("USER") or "p00"
PARTICIPANT = "".join(c for c in PARTICIPANT if c.isalnum() or c in "-_")[:20] or "p00"

# Same topic this participant's kredivo-02.ipynb wrote into.
TOPIC_ENRICHED = f"orders_enriched_{PARTICIPANT}"



# The Iceberg REST Catalog Tableflow exposes for every environment. Deterministic from
# region/provider/org/env -- no separate API call needed to discover it.
ICEBERG_CATALOG_URI = (
    f"https://tableflow.{CLOUD_REGION}.{CLOUD_PROVIDER}.confluent.cloud"
    f"/iceberg/catalog/organizations/{ORG_ID}/environments/{ENV_ID}"
)

def cli_available():
    """True if the `confluent` CLI is installed and has a logged-in session."""
    if shutil.which("confluent") is None:
        return False, "the `confluent` CLI is not on PATH"
    try:
        r = subprocess.run(["confluent", "environment", "list", "-o", "json"],
                            capture_output=True, text=True, timeout=15)
    except Exception as e:
        return False, f"could not run `confluent`: {e}"
    if r.returncode != 0:
        return False, f"not logged in (or no access) -- run `confluent login`: {r.stderr.strip()[:200]}"
    return True, "logged in"

CLI_OK, CLI_DETAIL = cli_available()

print(f"{INFO} Participant namespace     : {PARTICIPANT}")
print(f"{INFO} Source topic               : {TOPIC_ENRICHED}")
print(f"{INFO} Iceberg REST Catalog URI   : {ICEBERG_CATALOG_URI}")
print(f"{OK if CLI_OK else WARN} Confluent CLI usable for enable/status/disable steps: {CLI_DETAIL}")
print(f"{OK if TABLEFLOW_QUERY_AVAILABLE else WARN} Tableflow query credentials available: {TABLEFLOW_QUERY_AVAILABLE}"
      + ("" if TABLEFLOW_QUERY_AVAILABLE else " -- query cells below will print the catalog URI and "
         "code to run once TABLEFLOW_API_KEY / TABLEFLOW_API_SECRET are set in .env."))

## Step 1 -- Enable Tableflow on `orders_enriched_<you>`

Enabling Tableflow is a control-plane operation -- there's no data-plane REST call worth hand
rolling here, so we drive it with the same tool a platform engineer would use:
`confluent tableflow topic enable`. We choose:

- `--storage-type MANAGED` -- Confluent-managed object storage. No S3/ADLS bucket of our own to
  provision or grant access to; Confluent owns the files. (The alternative, `BYOS`, materializes
  into a bucket you own, which is the right choice when a data platform team wants the raw Parquet
  files under their own retention/lifecycle policy.)
- `--table-formats ICEBERG` -- the other option is `DELTA`. We pick Iceberg because every engine
  we query from below (DuckDB, and by reference Snowflake/Athena/Trino/Spark) speaks the Iceberg
  REST Catalog protocol natively.

In [ ]:
RUN_IN_NOTEBOOK = True  # Set to False if you'd rather copy the command below and run it yourself in a terminal

ENABLE_CMD = [
    "confluent", "tableflow", "topic", "enable", TOPIC_ENRICHED,
    "--cluster", CLUSTER_ID,
    "--environment", ENV_ID,
    "--storage-type", "MANAGED",
    "--table-formats", "ICEBERG",
    "-o", "json",
]

print(f"$ {' '.join(ENABLE_CMD)}\n")

if not RUN_IN_NOTEBOOK:
    print(f"{INFO} RUN_IN_NOTEBOOK = False -- run the command above yourself in a terminal, then continue "
          f"to the next cell to check status.")
elif not CLI_OK:
    print(f"{WARN} Skipping execution ({CLI_DETAIL}). The command above is exactly what would run --\n"
          f"     install/login to the Confluent CLI and re-run this cell to actually enable Tableflow.")
else:
    r = subprocess.run(ENABLE_CMD, capture_output=True, text=True, timeout=60)
    if r.returncode == 0:
        print(f"{OK} Tableflow enabled on {TOPIC_ENRICHED}.")
        try:
            print(pyjson.dumps(pyjson.loads(r.stdout), indent=2))
        except Exception:
            print(r.stdout)
    else:
        # Enabling twice on an already-enabled topic returns a non-zero exit -- treat that as
        # informational, not a failure, so this cell is safe to re-run.
        if "already" in (r.stderr + r.stdout).lower():
            print(f"{INFO} Tableflow already enabled on {TOPIC_ENRICHED} -- continuing.")
        else:
            print(f"{FAIL} {r.stderr.strip() or r.stdout.strip()}")

## Step 2 -- Wait for the table to materialize

The very first sync after enabling Tableflow has to snapshot the topic's existing history into
Parquet + Iceberg metadata, which takes longer than the steady-state incremental syncs that follow
(typically single-digit minutes for a topic this small). We poll `confluent tableflow topic
describe` until its status is no longer `PENDING`.

In [ ]:
RUN_IN_NOTEBOOK = True  # Set to False if you'd rather copy the command below and run it yourself in a terminal

DESCRIBE_CMD = [
    "confluent", "tableflow", "topic", "describe", TOPIC_ENRICHED,
    "--cluster", CLUSTER_ID,
    "--environment", ENV_ID,
    "-o", "json",
]

print(f"$ {' '.join(DESCRIBE_CMD)}\n")

if not RUN_IN_NOTEBOOK:
    print(f"{INFO} RUN_IN_NOTEBOOK = False -- run the command above yourself in a terminal to check status "
          f"(re-run it until the status is no longer PENDING).")
elif not CLI_OK:
    print(f"{WARN} Skipping execution ({CLI_DETAIL}). Run the command above yourself to check status.")
else:
    deadline = time.time() + 180
    last = None
    while time.time() < deadline:
        r = subprocess.run(DESCRIBE_CMD, capture_output=True, text=True, timeout=30)
        if r.returncode != 0:
            print(f"{FAIL} {r.stderr.strip() or r.stdout.strip()}")
            break
        info = pyjson.loads(r.stdout)
        status = (info.get("status") or {}).get("phase") or info.get("status")
        if status != last:
            print(f"{INFO} status = {status}")
            last = status
        if status not in ("PENDING", "PROVISIONING", None):
            print(f"\n{OK if status in ('RUNNING', 'ACTIVE', 'SYNCING') else WARN} Tableflow status: {status}")
            print(pyjson.dumps(info, indent=2))
            break
        time.sleep(10)
    else:
        print(f"{WARN} Still not past PENDING after 3 minutes -- re-run this cell; large first syncs can take longer.")

## Step 3 -- Query it from this notebook: `pyiceberg`

This is the same access path any Python-based analytics job would use -- no Confluent CLI, no
Kafka client, just the Iceberg REST Catalog protocol authenticated with the Tableflow-scoped API
key from `.env`.

Confluent's catalog uses **basic auth** (`credential = "key:secret"`), the same shape as the Kafka
and Schema Registry keys in Part 1/2 -- just a different key, scoped to Tableflow rather than the
Kafka cluster. The catalog's `warehouse` is the Kafka cluster ID; the namespace Tableflow tables
live under is that same cluster ID again; the table name is the topic name.

In [ ]:
PYICEBERG_SNIPPET = (
    "from pyiceberg.catalog.rest import RestCatalog\n\n"
    "catalog = RestCatalog(\n"
    "    name=\"tableflow\",\n"
    f"    uri=\"{ICEBERG_CATALOG_URI}\",\n"
    f"    warehouse=\"{CLUSTER_ID}\",\n"
    "    credential=\"<TABLEFLOW_API_KEY>:<TABLEFLOW_API_SECRET>\",\n"
    ")\n"
    f"table = catalog.load_table(\"{CLUSTER_ID}.{TOPIC_ENRICHED}\")\n"
    "df = table.scan().to_pandas()\n"
)

if not TABLEFLOW_QUERY_AVAILABLE:
    print(f"{WARN} No Tableflow query credentials -- here's what you'd run once TABLEFLOW_API_KEY / "
          f"TABLEFLOW_API_SECRET are set in .env:\n")
    print(PYICEBERG_SNIPPET)
else:
    from pyiceberg.catalog.rest import RestCatalog

    catalog = RestCatalog(
        name="tableflow",
        uri=ICEBERG_CATALOG_URI,
        warehouse=CLUSTER_ID,
        credential=f"{TABLEFLOW_API_KEY}:{TABLEFLOW_API_SECRET}",
    )
    try:
        print(f"{INFO} Namespaces visible to this key: {list(catalog.list_namespaces())}")
        table = catalog.load_table(f"{CLUSTER_ID}.{TOPIC_ENRICHED}")
        print(f"{OK} Loaded table {CLUSTER_ID}.{TOPIC_ENRICHED} -- current snapshot: {table.current_snapshot()}")
        df = table.scan().to_pandas()
        print(f"{OK} Read {len(df)} rows via the Iceberg REST Catalog (no Kafka client involved).")
        display(df.head(10))
    except Exception as e:
        print(f"{WARN} Could not query via pyiceberg: {e}")
        print(f"     Check: is the key scoped to environment {ENV_ID}? Has Step 2's first sync finished?")

## Step 4 -- Query it from DuckDB

This is the check a data analyst on their own laptop would run -- no Python client library, no
Confluent CLI, just `duckdb`'s `iceberg` extension pointed at the same catalog endpoint and key.
The SQL below is exactly what you'd paste into the `duckdb` CLI or DBeaver once you have a
Tableflow API key; we also run it here via the `duckdb` Python package so it's a live check, not
just documentation.

In [ ]:
DUCKDB_SQL = f"""
INSTALL iceberg;
LOAD iceberg;

CREATE SECRET tableflow_secret (
    TYPE ICEBERG,
    TOKEN '<bearer-token-fetched-via-oauth2-client-credentials>',
    ENDPOINT '{ICEBERG_CATALOG_URI}'
);

ATTACH '{CLUSTER_ID}' AS tableflow_catalog (
    TYPE ICEBERG,
    SECRET tableflow_secret,
    ENDPOINT '{ICEBERG_CATALOG_URI}'
);

SELECT category, COUNT(*) AS orders, ROUND(SUM(amount), 2) AS total_amount
FROM tableflow_catalog."{CLUSTER_ID}"."{TOPIC_ENRICHED}"
GROUP BY category
ORDER BY total_amount DESC;
"""

print(DUCKDB_SQL)

if not TABLEFLOW_QUERY_AVAILABLE:
    print(f"{WARN} No Tableflow query credentials -- the SQL above is what to run once TABLEFLOW_API_KEY / "
          f"TABLEFLOW_API_SECRET are set in .env.")
else:
    import duckdb

    # DuckDB's built-in OAuth2 CLIENT_ID/CLIENT_SECRET flow hardcodes a Polaris-style default
    # scope (`PRINCIPAL_ROLE:ALL`) that Confluent's Tableflow catalog rejects with
    # `invalid_scope - unsupported scope: PRINCIPAL_ROLE:ALL`. There's no CREATE SECRET knob to
    # override that scope, so instead we fetch the bearer token ourselves (the same
    # client_credentials request DuckDB would make, but with the `catalog` scope pyiceberg
    # defaults to -- Confluent's catalog requires *some* scope to be present, just not
    # PRINCIPAL_ROLE:ALL) and hand DuckDB a plain TOKEN, skipping its OAuth2 negotiation entirely.
    token_resp = requests.post(
        f"{ICEBERG_CATALOG_URI}/v1/oauth/tokens",
        data={"grant_type": "client_credentials", "scope": "catalog"},
        auth=(TABLEFLOW_API_KEY, TABLEFLOW_API_SECRET),
    )

    con = duckdb.connect()
    try:
        token_resp.raise_for_status()
        access_token = token_resp.json()["access_token"]

        con.execute("INSTALL iceberg; LOAD iceberg;")
        con.execute(f"""
            CREATE OR REPLACE SECRET tableflow_secret (
                TYPE ICEBERG,
                TOKEN '{access_token}',
                ENDPOINT '{ICEBERG_CATALOG_URI}'
            );
        """)
        con.execute(f"""
            ATTACH '{CLUSTER_ID}' AS tableflow_catalog (
                TYPE ICEBERG,
                SECRET tableflow_secret,
                ENDPOINT '{ICEBERG_CATALOG_URI}'
            );
        """)
        result = con.execute(f"""
            SELECT category, COUNT(*) AS orders, ROUND(SUM(amount), 2) AS total_amount
            FROM tableflow_catalog."{CLUSTER_ID}"."{TOPIC_ENRICHED}"
            GROUP BY category
            ORDER BY total_amount DESC;
        """).fetchdf()
        print(f"\n{OK} Queried via DuckDB's Iceberg REST Catalog support:")
        display(result)
    except requests.HTTPError as e:
        print(f"\n{WARN} OAuth2 token request failed: {e} -- {token_resp.text[:300]}")
    except Exception as e:
        print(f"\n{WARN} DuckDB attach/query failed: {e}")
        print(f"     DuckDB's Iceberg REST Catalog auth options have moved fast across versions -- check "
              f"`duckdb.org/docs/extensions/iceberg` for the current secret syntax if this keeps failing.")
    finally:
        con.close()

## Step 5 -- Access from other tools (reference)

We don't have live Snowflake/AWS/Databricks accounts wired into this workshop, so these aren't
executed -- they're the exact connection shape for each engine, all pointed at the **same** Iceberg
table Steps 3-4 just read. Nothing about the table changes per consumer; only the catalog
registration differs.

### Snowflake

Snowflake reads Tableflow tables as **externally-managed Iceberg tables**, either against
Confluent's built-in REST catalog directly, or against Snowflake Open Catalog/Polaris if you've set
up a catalog integration in the Tableflow console. Direct-to-Confluent-catalog looks like:

```sql
CREATE CATALOG INTEGRATION tableflow_catalog_int
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  CATALOG_NAMESPACE = '<cluster-id>'
  REST_CONFIG = (
    CATALOG_URI = '<ICEBERG_CATALOG_URI>'
    CATALOG_NAME = '<cluster-id>'
  )
  REST_AUTHENTICATION = (
    TYPE = BEARER
    BEARER_TOKEN = '<tableflow-api-key>:<tableflow-api-secret>'
  )
  ENABLED = TRUE;

CREATE ICEBERG TABLE orders_enriched
  CATALOG = 'tableflow_catalog_int'
  CATALOG_TABLE_NAME = '<topic-name>';

SELECT * FROM orders_enriched LIMIT 10;
```

### Amazon Athena / Trino

Both speak the Iceberg REST Catalog spec (Trino via its `iceberg` connector, `iceberg.rest-catalog`
type; Athena via a Glue-federated or REST-catalog-backed workgroup). Trino `catalog.properties`:

```properties
connector.name=iceberg
iceberg.catalog.type=rest
iceberg.rest-catalog.uri=<ICEBERG_CATALOG_URI>
iceberg.rest-catalog.warehouse=<cluster-id>
iceberg.rest-catalog.security=OAUTH2
iceberg.rest-catalog.oauth2.credential=<tableflow-api-key>:<tableflow-api-secret>
```

```sql
SELECT category, COUNT(*) FROM tableflow."<cluster_id>"."<topic-name>" GROUP BY category;
```

Alternatively, if you set up a **catalog integration to AWS Glue** in the Tableflow console (Part
5's Step 1 could target `BYOS` + Glue instead of `MANAGED`), Athena reads the table as a normal
Glue-cataloged Iceberg table with no REST catalog config at all -- just `SELECT * FROM
<glue_db>.<topic_name>`.

### Apache Spark

```python
spark = (
    SparkSession.builder
    .config("spark.sql.catalog.tableflow", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.tableflow.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config("spark.sql.catalog.tableflow.uri", ICEBERG_CATALOG_URI)
    .config("spark.sql.catalog.tableflow.warehouse", CLUSTER_ID)
    .config("spark.sql.catalog.tableflow.credential", f"{TABLEFLOW_API_KEY}:{TABLEFLOW_API_SECRET}")
    .getOrCreate()
)
spark.sql(f'SELECT * FROM tableflow."{CLUSTER_ID}"."{TOPIC_ENRICHED}" LIMIT 10').show()
```

The common thread across all four: **one Iceberg table, one REST catalog endpoint, credentials
scoped per consumer** -- the same governance model as the Kafka/Schema-Registry/Flink keys already
in `.env`, just extended one hop further downstream than Part 2 went.

## Cleanup

Set `CLEANUP = True` and pick a `CLEANUP_MODE` below, then re-run this cell:

- **`"participant"` (default, safer):** deletes only resources belonging to this participant --
  the topics/subjects namespaced `<...>_<you>` from Parts 1 and 2, plus Flink statements whose
  name contains your participant namespace, plus disabling Tableflow on `orders_enriched_<you>`.
  Does not touch anyone else's data.
- **`"all"` (unscoped, destructive):** ignores participant namespacing entirely -- deletes
  **every** topic on `CONFLUENT_CLUSTER_ID`, **every** schema subject in this Schema Registry
  environment, and **every** Flink statement in this environment, then disables Tableflow on
  `orders_enriched_<you>`. If this cluster/environment is shared with other workshop attendees,
  their topics, schemas, and Flink statements are deleted too. Only use this on a
  cluster/environment that's entirely yours, or when you intend to reset it for everyone.

This cell rebuilds its own Kafka/Schema Registry clients from `.env` rather than reusing
objects from another notebook's kernel, since Parts 1-3 normally run in separate Jupyter
sessions. Safe to re-run -- deleting an already-deleted topic/subject/statement is reported,
not treated as fatal.

In [ ]:
CLEANUP = False
CLEANUP_MODE = "participant"  # "participant" (safe, scoped to you) or "all" (deletes everything -- see markdown above)
RUN_IN_NOTEBOOK = True  # Set to False if you'd rather copy DISABLE_CMD and run it yourself in a terminal

assert CLEANUP_MODE in ("participant", "all"), 'CLEANUP_MODE must be "participant" or "all"'

DISABLE_CMD = [
    "confluent", "tableflow", "topic", "disable", TOPIC_ENRICHED,
    "--cluster", CLUSTER_ID,
    "--environment", ENV_ID,
    "--force",
]

# Part 1 and Part 2 topic/subject names, derived the same way their own notebooks derive them.
# Only used in "participant" mode.
TOPIC_BNPL = f"bnpl_transactions_{PARTICIPANT}"
TOPIC_BNPL_SECURE = f"{TOPIC_BNPL}_secure"

TOPIC_ORDERS = f"orders_source_{PARTICIPANT}"
TOPIC_DUP_STATS = f"orders_dup_stats_{PARTICIPANT}"
TOPIC_DEDUPED = f"orders_deduped_{PARTICIPANT}"
TOPIC_FILTERED = f"orders_high_value_{PARTICIPANT}"
TOPIC_CATALOG = f"product_catalog_{PARTICIPANT}"
# TOPIC_ENRICHED already defined above (Part 3's own source topic == Part 2's output).

PARTICIPANT_TOPICS = [
    TOPIC_BNPL, TOPIC_BNPL_SECURE,
    TOPIC_ORDERS, TOPIC_DUP_STATS, TOPIC_DEDUPED, TOPIC_FILTERED, TOPIC_CATALOG, TOPIC_ENRICHED,
]
PARTICIPANT_SUBJECTS = [
    f"{TOPIC_BNPL}-value", f"{TOPIC_BNPL_SECURE}-value",
    f"{TOPIC_ORDERS}-value", f"{TOPIC_CATALOG}-value",
]

if not CLEANUP:
    print(f"{INFO} Cleanup disabled. Set CLEANUP = True and re-run to tear down (mode = '{CLEANUP_MODE}'):")
    print(f"     Tableflow : $ {' '.join(DISABLE_CMD)}")
    if CLEANUP_MODE == "participant":
        print(f"     topics    : {', '.join(PARTICIPANT_TOPICS)}")
        print(f"     subjects  : {', '.join(PARTICIPANT_SUBJECTS)}")
        print(f"     Flink     : statements matching '{PARTICIPANT}'")
    else:
        print(f"     topics    : ALL topics on cluster {CLUSTER_ID}")
        print(f"     subjects  : ALL subjects in this Schema Registry")
        print(f"     Flink     : ALL statements in this environment")
        print(f"     {WARN} 'all' mode is unscoped -- it deletes other participants' resources too "
              f"if this cluster/environment is shared.")
else:
    # -- Part 3: disable Tableflow --
    if not RUN_IN_NOTEBOOK:
        print(f"{INFO} RUN_IN_NOTEBOOK = False -- run this yourself in a terminal:\n")
        print(f"     $ {' '.join(DISABLE_CMD)}")
    elif not CLI_OK:
        print(f"{WARN} Cannot run Tableflow disable ({CLI_DETAIL}). Run the command above yourself.")
    else:
        r = subprocess.run(DISABLE_CMD, capture_output=True, text=True, timeout=60)
        if r.returncode == 0:
            print(f"{OK} Tableflow disabled on {TOPIC_ENRICHED}.")
        else:
            print(f"{FAIL} {r.stderr.strip() or r.stdout.strip()}")

    # -- Rebuild Kafka + Schema Registry clients --
    from confluent_kafka.admin import AdminClient
    from confluent_kafka.schema_registry import SchemaRegistryClient

    bootstrap_servers = os.getenv("CCLOUD_BOOTSTRAP_SERVERS")
    kafka_api_key = os.getenv("CCLOUD_API_KEY")
    kafka_api_secret = os.getenv("CCLOUD_API_SECRET")
    sr_url = os.getenv("SCHEMA_REGISTRY_URL")
    sr_api_key = os.getenv("SCHEMA_REGISTRY_API_KEY")
    sr_api_secret = os.getenv("SCHEMA_REGISTRY_API_SECRET")

    if not all([bootstrap_servers, kafka_api_key, kafka_api_secret, sr_url, sr_api_key, sr_api_secret]):
        print(f"{WARN} Kafka/Schema Registry credentials missing from .env -- cannot clean up topics/subjects.")
    else:
        admin = AdminClient({
            "bootstrap.servers": bootstrap_servers,
            "security.protocol": "SASL_SSL",
            "sasl.mechanisms": "PLAIN",
            "sasl.username": kafka_api_key,
            "sasl.password": kafka_api_secret,
        })
        sr_client = SchemaRegistryClient({
            "url": sr_url,
            "basic.auth.user.info": f"{sr_api_key}:{sr_api_secret}",
        })

        flink_api_key = os.getenv("FLINK_API_KEY", "")
        flink_api_secret = os.getenv("FLINK_API_SECRET", "")
        flink_rest_available = all([CLOUD_REGION, CLOUD_PROVIDER, ORG_ID, ENV_ID, flink_api_key, flink_api_secret])
        flink_rest_base = (
            f"https://flink.{CLOUD_REGION}.{CLOUD_PROVIDER}.confluent.cloud"
            f"/sql/v1/organizations/{ORG_ID}/environments/{ENV_ID}/statements"
        )

        # -- Flink statements --
        if flink_rest_available:
            try:
                r = requests.get(flink_rest_base, auth=(flink_api_key, flink_api_secret), timeout=30)
                r.raise_for_status()
                all_statements = r.json().get("data", [])
                if CLEANUP_MODE == "all":
                    names = [s["name"] for s in all_statements]
                else:
                    names = [s["name"] for s in all_statements
                             if PARTICIPANT.lower() in str(s.get("name", "")).lower()]
                for n in names:
                    rr = requests.delete(f"{flink_rest_base}/{n}", auth=(flink_api_key, flink_api_secret), timeout=30)
                    print(f"{OK if rr.status_code in (200, 202, 204) else FAIL} statement {n}")
                if not names:
                    print(f"{INFO} No matching Flink statements.")
            except Exception as e:
                print(f"{WARN} Flink cleanup: {e}")
        else:
            print(f"{INFO} Flink REST credentials (CLOUD_REGION/CLOUD_PROVIDER/ORG_ID/ENV_ID/FLINK_API_KEY/FLINK_API_SECRET) not fully set -- skipping Flink statement cleanup.")

        # -- Schema subjects --
        if CLEANUP_MODE == "all":
            try:
                subjects_to_delete = sr_client.get_subjects()
            except Exception as e:
                print(f"{WARN} Could not list subjects: {e}")
                subjects_to_delete = []
        else:
            subjects_to_delete = PARTICIPANT_SUBJECTS

        for subj in subjects_to_delete:
            for permanent in (False, True):
                try:
                    sr_client.delete_subject(subj, permanent=permanent)
                    print(f"{OK} subject {subj} ({'hard' if permanent else 'soft'} delete)")
                except Exception as e:
                    print(f"{INFO} subject {subj} ({'hard' if permanent else 'soft'}): {str(e)[:90]}")

        # -- Topics --
        if CLEANUP_MODE == "all":
            try:
                cluster_md = admin.list_topics(timeout=15)
                topics_to_delete = [t for t in cluster_md.topics.keys() if not t.startswith("_")]
            except Exception as e:
                print(f"{WARN} Could not list topics: {e}")
                topics_to_delete = []
        else:
            topics_to_delete = PARTICIPANT_TOPICS

        for name, fut in admin.delete_topics(topics_to_delete).items():
            try:
                fut.result(timeout=60)
                print(f"{OK} topic {name} deleted")
            except Exception as e:
                print(f"{INFO} topic {name}: {str(e)[:90]}")

    print(f"\n{OK} Cleanup complete (mode = '{CLEANUP_MODE}').")